# QFT repeated secret

**Run All** to see who acts first, how much Eve learns, and how often Bob accepts a key that Eve recovered.

[Glossary of settings and result columns](validation/MODEL_NOTES.md#glossary)

## Configuration

In [ ]:
ATTEMPTS = 100  # Cap on transmissions of the four-bit secret.
TRIALS = 1024  # Independent trials each for fit, held-out noise, and attack.
BLOCK_QUBITS = 2  # Choose 2 or 4. Half of each block is fresh hidden checks.
INTERCEPT_PROBABILITY = 0.025  # Per Eve, per qubit, on each leg.
EVE_Z_PROBABILITIES = (0.1, 0.9)  # The two Eves favor different bases.
COALITION_SIZES = (1, 2, 4, 8, 16)  # All Eves use X in the alarm policy comparison.
RANDOM_LOOK_RANGE = (2, 4)  # How many chances Bob has to alarm, including the final attempt.
RUN_ALARM_COMPARISON = True  # Set False to skip this expansion for long caps.
COMPARE_PREPARED_STATE = True  # Include QFT's weighted check-error score.
Q_Z, Q_X = 0.002, 0.038  # Per-leg Pauli X and Z probabilities.
SEED = 20261122

## 1. Does Bob alarm before Eve recovers the key?

All enabled detectors see the same transmissions. The left panel shows how often Bob alarms when there is no attack. The right panel compares Bob's alarm with Eve's first guess during an attack. Eve guesses when a key reaches at least 99% confidence. An alarm on the same attempt as her guess counts for Bob.

With `COMPARE_PREPARED_STATE = True`, the third detector adapts the proposal's weighted error score to QFT. It uses each check's noise-only error probability based on the revealed check values and positions, Bob's phases, and the known noise rates. It averages over the unknown payload. Set it to `False` to use the pooled and joint check scores.

In [ ]:
from importlib import reload

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from quantum_study import qft_exchange, qft_study

# Pick up local code changes when rerunning after a pull.
reload(qft_exchange)
reload(qft_study)

summary, curves = qft_study.run_qft_study(
    attempts=ATTEMPTS, trials=TRIALS, block_qubits=BLOCK_QUBITS,
    intercept_probability=INTERCEPT_PROBABILITY,
    eve_z_probabilities=EVE_Z_PROBABILITIES, q_z=Q_Z, q_x=Q_X, seed=SEED,
    compare_prepared_state=COMPARE_PREPARED_STATE,
)
summary = summary.set_index("detector")
outcomes = summary[[
    "alarm_rate", "correct_recovery_rate", "incorrect_recovery_rate", "inconclusive_rate",
]].copy()
outcomes.columns = ["Bob alarmed first", "Eve recovered", "Eve guessed wrong", "Reached attempt cap"]

fig, (null_ax, attack_ax) = plt.subplots(1, 2, sharey=True, figsize=(10, 3.5), layout="constrained")
false_alarms = summary["false_alarm_rate"]
bars = null_ax.barh(summary.index, false_alarms, height=0.5, color="#4C78A8")
null_ax.bar_label(bars, labels=[f"{rate:.1%}" for rate in false_alarms], padding=4,
                  bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.2})
null_ax.set_xlim(0, max(0.08, false_alarms.max() * 1.25))
null_ax.axvline(0.05, color="0.3", linestyle="--")
null_ax.text(0.05, 0.98, "5% target", transform=null_ax.get_xaxis_transform(),
             ha="center", va="top", fontsize=9, backgroundcolor="white")
null_ax.set(title="Noise only · false alarms", xlabel="Fraction of trials", ylabel="")
outcomes.plot.barh(ax=attack_ax, stacked=True, color=["#4C78A8", "#14866D", "#D56C36", "#C9D4DB"])
attack_ax.set(title="During an attack, who went first?", xlabel="Fraction of trials", ylabel="", xlim=(0, 1))
attack_ax.tick_params(labelleft=False)
alarm_labels = [f"{rate:.1%}" if rate >= 0.08 else "" for rate in outcomes["Bob alarmed first"]]
attack_ax.bar_label(attack_ax.containers[0], labels=alarm_labels, label_type="center", color="white")
attack_ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.23), ncol=2)
for axis in (null_ax, attack_ax):
    axis.xaxis.set_major_formatter(PercentFormatter(1))
plt.show()

for detector, row in summary.iterrows():
    recovered = round(row["correct_recovery_rate"] * TRIALS)
    print(f"{detector}: Eve recovered the key before Bob's alarm in {recovered}/{TRIALS} trials.")

## 2. Of Bob's accepted keys, how many are exposed?

Bob accepts the key if he never alarms and his final decoded key matches Alice's.
An accepted key is exposed if Eve guessed it correctly. Eve stops intercepting
after her guess. Bob keeps checking until he alarms or reaches the attempt cap.
Each row counts attacked sessions.

If Alice and Bob use an exposed accepted key to encrypt traffic Eve can observe,
she can use that key to decrypt it while listening passively.


In [ ]:
accepted_key_labels = {
    "accepted_sessions": "Accepted sessions",
    "exposed_accepted_keys": "Exposed accepted keys",
    "accepted_key_exposure_rate": "Exposure among accepted keys",
}
if "summary" not in globals() or not set(accepted_key_labels).issubset(summary.columns):
    raise RuntimeError(
        "Run the notebook from the top to generate the accepted-key summary. "
        "If this persists, check that the notebook and study code are both up to date."
    )

accepted_keys = summary[list(accepted_key_labels)].rename(columns=accepted_key_labels)
accepted_keys["Exposure among accepted keys"] = (
    summary["accepted_key_exposure_rate"].map("{:.1%}".format, na_action="ignore")
    .astype("string")
    .fillna("Undefined (0 accepted)")
)
display(accepted_keys)


## 3. How does Eve's knowledge of the secret grow?

The upper curves show the average probability Eve assigns to each **true** bit value. Each trial's confidence stays at its last value once Bob alarms or Eve guesses the key.

The lower line shows the percentage of attacked trials where Eve had recovered the **full key** by that attempt. It counts recoveries before Bob's alarm, including trials where Bob never alarms.

In [ ]:
bits = curves[[f"bit_{bit}" for bit in range(1, 5)]]
lower = max(0, min(0.48, bits.min().min() - 0.02))
upper = min(1, max(0.55, bits.max().max() + 0.02))
recovery = "cumulative_correct_recovery_rate"
recovery_upper = min(1, max(0.05, curves[recovery].max() * 1.1))
detector_groups = list(curves.groupby("detector", sort=False))
detector_count = len(detector_groups)
fig, axes = plt.subplots(
    2, detector_count, sharex="col", sharey="row",
    figsize=(max(10, 4.3 * detector_count), 5.4),
    height_ratios=(2, 1), layout="constrained", squeeze=False,
)
bit_axes, recovery_axes = axes
for (detector, group), bit_ax, recovery_ax in zip(
    detector_groups, bit_axes, recovery_axes,
):
    for bit in range(1, 5):
        bit_ax.plot(group["attempt"], group[f"bit_{bit}"], label=f"Bit {bit}")
    bit_ax.axhline(0.5, color="0.5", linestyle="--", linewidth=1)
    bit_ax.set(title=detector, ylim=(lower, upper))
    bit_ax.yaxis.set_major_formatter(PercentFormatter(1))
    recovery_ax.step(group["attempt"], group[recovery], where="post", color="#14866D", zorder=3)
    recovery_ax.set(xlabel="Attempt", ylim=(0, recovery_upper))
    recovery_ax.yaxis.set_major_formatter(PercentFormatter(1))
bit_axes[0].set_ylabel("Mean probability of true bit")
bit_axes[-1].legend(loc="upper left", ncol=2)
recovery_axes[0].set_ylabel("Trials with full recovery")
plt.show()

## 4. What happens if Bob dismisses an alarm?

On the fixed schedule, Bob can alarm at 25%, 50%, and 100% of the attempt cap.
The random schedule chooses 2–4 attempts by default and always includes the
final attempt. The 5% target is for at least one alarm in a noise-only trial.

**Reject on alarm** discards the key as soon as Bob alarms. **Reset on early
alarm** lets Bob dismiss an alarm before the final attempt and clear his
accumulated evidence. Alice keeps sending the same key and Eve keeps what she
has learned. Bob uses the same alarm rule on the evidence he collects after
resetting. Both policies reject the key if Bob alarms on the final attempt
or his key differs from Alice's.

The two policy columns show **exposed keys / accepted sessions**. The last
column counts exposed keys Bob accepted in trials where he reset. Eve may
have recovered the key before or after that reset. The noise-only table
counts any alarm, even if Bob dismisses it, and shows how often Bob rejects
the key because of an alarm. This follow-up keeps its original pooled and
joint-pattern detectors. The prepared-state option applies to sections 1–3.


In [ ]:
if RUN_ALARM_COMPARISON:
    from importlib import reload
    from quantum_study import qft_alarm

    reload(qft_alarm)
    alarm_comparison = qft_alarm.run_qft_alarm_comparison(
        attempts=ATTEMPTS, trials=TRIALS, block_qubits=BLOCK_QUBITS,
        intercept_probability=INTERCEPT_PROBABILITY,
        coalition_sizes=COALITION_SIZES, random_look_range=RANDOM_LOOK_RANGE,
        q_z=Q_Z, q_x=Q_X, seed=SEED,
    )

    comparison = alarm_comparison.replace({
        "schedule": {"Percentage": "Fixed"},
        "alarm_policy": {"Reset on interim alarm": "Reset on early alarm"},
    })
    comparison_index = ["eve_count", "effective_intercept_probability", "schedule", "detector"]
    policies = comparison.set_index(comparison_index)
    reject = policies[policies["alarm_policy"] == "Reject on alarm"]
    reset = policies[policies["alarm_policy"] == "Reset on early alarm"]
    policy_table = reject[["attack_crossing_rate", "recovered_before_first_crossing"]].copy()
    for label, outcome in (("Reject on alarm", reject), ("Reset on early alarm", reset)):
        accepted = outcome["accepted_sessions"]
        counts = outcome["exposed_accepted_keys"].astype("string") + "/" + accepted.astype("string")
        policy_table[label] = counts.where(accepted > 0, "0/0 (undefined)")
    policy_table["Exposed keys accepted after reset"] = reset["exposed_after_reset"]
    policy_table = policy_table.reset_index().rename(columns={
        "eve_count": "Eves",
        "effective_intercept_probability": "Effective intercept",
        "schedule": "Schedule",
        "detector": "Detector",
        "attack_crossing_rate": "Trials with an alarm",
        "recovered_before_first_crossing": "Eve recovered before alarm (count)",
    })
    for column in ("Effective intercept", "Trials with an alarm"):
        policy_table[column] = policy_table[column].map("{:.1%}".format)
    display(policy_table)

    noise_labels = {
        "schedule": "Schedule",
        "detector": "Detector",
        "alarm_policy": "Policy",
        "noise_crossing_rate": "Trials with an alarm",
        "noise_reject_rate": "Trials rejected by alarm",
    }
    noise_table = (
        comparison[list(noise_labels)].drop_duplicates()
        .rename(columns=noise_labels).copy()
    )
    for column in ("Trials with an alarm", "Trials rejected by alarm"):
        noise_table[column] = noise_table[column].map("{:.1%}".format)
    display(noise_table)
else:
    print("Alarm policy comparison skipped.")


## Method and sources

- **Circuit.** Bob scrambles private phases, Alice encodes the secret with fresh hidden checks, and Bob reverses the phases and applies the inverse QFT. [Papadopoulos and Linvill, §4.1 Eqs. (1)–(3) and §6.1](https://arxiv.org/abs/2404.12507v2)
- **Experiment.** Proposal §2 supplies the repeated four-bit key, per-leg Pauli model, and recovery threshold. The prepared-state score adapts the proposal's weighted error formula to QFT checks. Bob makes alarm decisions at 25%, 50%, and 100% of the attempt cap. The reference rates come from Erven et al. [§4](https://arxiv.org/abs/0901.0960v2)
- **Inference.** Averaging Bob's uniform private phases gives an exact mixed starting state for Eve's record likelihood. Bob's calculation still uses his actual phases. [Exchange code](quantum_study/qft_exchange.py) · [Study code](quantum_study/qft_study.py)
- **Alarm response.** Randomly spaced decisions and the option to dismiss an early alarm are follow-up policies for the repeated-secret study. [Alarm response code](quantum_study/qft_alarm.py)